In [ ]:
# 1. ENVIRONMENT SETUP & DEPENDENCIES
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from skimage.feature import hog
from skimage import exposure

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.model_selection import StratifiedKFold

# Set deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

# 2. DATASET INGESTION & ROBUST STRATIFIED SPLIT
# Kaggle Dataset Structure: chest_xray/{train, test, val}/{NORMAL, PNEUMONIA}
DATA_DIR = "chest_xray"

def compile_dataset_dataframe(base_dir):
    records = []
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            continue
        for label_name in ['NORMAL', 'PNEUMONIA']:
            class_folder = os.path.join(split_path, label_name)
            if not os.path.exists(class_folder):
                continue
            label = 0 if label_name == 'NORMAL' else 1
            for fname in os.listdir(class_folder):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    subgroup = 'bacteria' if 'bacteria' in fname.lower() else ('virus' if 'virus' in fname.lower() else 'normal')
                    records.append({
                        'filepath': os.path.join(class_folder, fname),
                        'label': label,
                        'class_name': label_name,
                        'subgroup': subgroup,
                        'original_split': split
                    })
    df = pd.DataFrame(records)
    return df

df_meta = compile_dataset_dataframe(DATA_DIR)
print(f"Total dataset records: {len(df_meta)}")
print(df_meta['class_name'].value_counts())

# Stratified Holdout Splitting (80% Train, 10% Validation, 10% Test)
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df_meta, test_size=0.10, stratify=df_meta['label'], random_state=SEED)
train_df, val_df = train_test_split(train_df, test_size=0.1111, stratify=train_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train set: {len(train_df)}, Val set: {len(val_df)}, Test set: {len(test_df)}")

# 3. CLASSICAL ML PIPELINE: FEATURE EXTRACTION + SVM
def extract_handcrafted_features(filepath, img_size=(128, 128)):

    """Extracts HOG descriptors and spatial intensity statistics from grayscale images."""
    img = Image.open(filepath).convert('L').resize(img_size)
    arr = np.array(img, dtype=np.float32) / 255.0

    # Histogram of Oriented Gradients (HOG)
    hog_feats = hog(arr, orientations=8, pixels_per_cell=(16, 16),
                    cells_per_block=(1, 1), visualize=False)

    # First-order statistical descriptors (mean, std, skewness proxy)
    mean_val = np.mean(arr)
    std_val = np.std(arr)
    p25, p75 = np.percentile(arr, [25, 75])
    stats_feats = np.array([mean_val, std_val, p25, p75])

    return np.hstack([hog_feats, stats_feats])

print("Extracting features for classical pipeline...")
X_train_classic = np.array([extract_handcrafted_features(p) for p in train_df['filepath']])
y_train_classic = train_df['label'].values

X_test_classic = np.array([extract_handcrafted_features(p) for p in test_df['filepath']])
y_test_classic = test_df['label'].values

# Stratified 5-Fold Cross-Validation on Classical SVM
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_classic, y_train_classic)):
    svm_fold = SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=SEED)
    svm_fold.fit(X_train_classic[train_idx], y_train_classic[train_idx])
    preds = svm_fold.predict(X_train_classic[val_idx])
    cv_f1_scores.append(f1_score(y_train_classic[val_idx], preds))

print(f"5-Fold CV F1-Score (Classical SVM): {np.mean(cv_f1_scores):.4f} +/- {np.std(cv_f1_scores):.4f}")

# Train Final SVM Model
svm_model = SVC(kernel='rbf', C=1.0, probability=True, class_weight='balanced', random_state=SEED)
svm_model.fit(X_train_classic, y_train_classic)
svm_test_preds = svm_model.predict(X_test_classic)
svm_test_probs = svm_model.predict_proba(X_test_classic)[:, 1]

# 4. DEEP LEARNING PIPELINE: PYTORCH RESNET-18 TRANSFER LEARNING
class XRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]['filepath']
        label = self.df.iloc[idx]['label']
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)

dl_train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dl_val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = XRayDataset(train_df, transform=dl_train_transforms)
val_dataset = XRayDataset(val_df, transform=dl_val_transforms)
test_dataset = XRayDataset(test_df, transform=dl_val_transforms)

# Address class imbalance with WeightedRandomSampler
class_counts = train_df['label'].value_counts()
weights = [1.0 / class_counts[int(l)] for l in train_df['label']]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model Definition
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = True
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 1)
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# Training Loop with Early Stopping
best_val_loss = float('inf')
best_weights = None

for epoch in range(1, 11):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            probs = torch.sigmoid(outputs)
            val_preds.extend(probs.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = model.state_dict().copy()

    print(f"Epoch {epoch}/10 - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

model.load_state_dict(best_weights)

# Test Inference
model.eval()
resnet_test_probs, resnet_test_targets = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs).squeeze(1)
        probs = torch.sigmoid(outputs)
        resnet_test_probs.extend(probs.cpu().numpy())
        resnet_test_targets.extend(labels.numpy())

resnet_test_probs = np.array(resnet_test_probs)
resnet_test_preds = (resnet_test_probs >= 0.5).astype(int)


# 5. COMPARATIVE MODEL EVALUATION & METRICS
def compute_metrics(y_true, y_pred, y_probs):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall (Sensitivity)': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_probs)
    }

metrics_svm = compute_metrics(y_test_classic, svm_test_preds, svm_test_probs)
metrics_resnet = compute_metrics(resnet_test_targets, resnet_test_preds, resnet_test_probs)

comparison_df = pd.DataFrame([metrics_svm, metrics_resnet], index=['Classical SVM (HOG)', 'Deep Learning (ResNet-18)'])
print("\n=== COMPARATIVE PERFORMANCE ON UNSEEN TEST SET ===")
print(comparison_df.round(4))

# Confusion Matrix Visualisation
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(confusion_matrix(y_test_classic, svm_test_preds), annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title("Classical SVM Confusion Matrix")
ax[0].set_xlabel("Predicted")
ax[0].set_ylabel("True")

sns.heatmap(confusion_matrix(resnet_test_targets, resnet_test_preds), annot=True, fmt='d', cmap='Purples', ax=ax[1])
ax[1].set_title("ResNet-18 Confusion Matrix")
ax[1].set_xlabel("Predicted")
ax[1].set_ylabel("True")
plt.tight_layout()
plt.show()

# 6. EXPLAINABILITY: GRAD-CAM IMPLEMENTATION
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_layers()

    def hook_layers(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate_heatmap(self, input_tensor):
        self.model.eval()
        output = self.model(input_tensor).squeeze(1)
        self.model.zero_grad()
        output.backward()

        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations[0].detach()
        for i in range(activations.size(0)):
            activations[i, :, :] *= pooled_gradients[i]

        heatmap = torch.mean(activations, dim=0).cpu().numpy()
        heatmap = np.maximum(heatmap, 0)
        heatmap /= (np.max(heatmap) + 1e-8)
        return heatmap

grad_cam = GradCAM(model, model.layer4[-1])

# Generate diagnostic explainability plot for a sample test pneumonia scan
sample_idx = np.where(test_df['label'] == 1)[0][0]
sample_path = test_df.iloc[sample_idx]['filepath']
raw_img = Image.open(sample_path).convert('RGB').resize((224, 224))
input_tensor = dl_val_transforms(raw_img).unsqueeze(0).to(device)

cam_heatmap = grad_cam.generate_heatmap(input_tensor)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw_img)
axes[0].set_title("Input Chest Radiograph")
axes[0].axis('off')

axes[1].imshow(raw_img)
axes[1].imshow(cam_heatmap, cmap='jet', alpha=0.5)
axes[1].set_title("Grad-CAM Pathology Localization")
axes[1].axis('off')
plt.show()

# 7. SUBGROUP BIAS & DIAGNOSTIC SLICING
test_df['pred_resnet'] = resnet_test_preds

# Evaluate Performance Disparities by Pathological Subtype
print("\n=== SUBGROUP PERFORMANCE ANALYSIS ===")
for sub in ['bacteria', 'virus']:
    sub_df = test_df[test_df['subgroup'] == sub]
    sub_acc = accuracy_score(sub_df['label'], sub_df['pred_resnet'])
    sub_rec = recall_score(sub_df['label'], sub_df['pred_resnet'])
    print(f"Subgroup [{sub.upper()}] (n={len(sub_df)}) -> Recall: {sub_rec:.4f}, Accuracy: {sub_acc:.4f}")

Compute Device: cpu
Total dataset records: 0


KeyError: 'class_name'